<a href="https://colab.research.google.com/github/MatiasMoreno707/lab13-lp/blob/dev/lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd


In [3]:
df = pd.read_csv("garments_worker_productivity.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   quarter                1197 non-null   object 
 2   department             1197 non-null   object 
 3   day                    1197 non-null   object 
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: f

In [5]:
# Eliminar la columna 'date'
df = df.drop(columns=['date'])

# Crear variable binaria a partir de 'actual_productivity'
df['actual_productivity_class'] = (df['actual_productivity'] >= 0.5).astype(int)


In [6]:
df['actual_productivity_class'].value_counts()


,count
actual_productivity_class,
1,1064
0,133


In [7]:
# valores nulos
missing_values = df.isnull().sum().sort_values(ascending=False)
print(missing_values[missing_values > 0])


wip    506
dtype: int64


In [8]:
# Imputación con mediana
df['wip'].fillna(df['wip'].median(), inplace=True)


/tmp/ipython-input-8-4242153929.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['wip'].fillna(df['wip'].median(), inplace=True)


In [12]:
import numpy as np

# Identificar columnas numéricas
num_cols = df.select_dtypes(include=['float64', 'int64']).drop(columns=['actual_productivity', 'actual_productivity_class']).columns

# Eliminar outliers univariados usando IQR
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]


In [13]:
from sklearn.ensemble import IsolationForest

# Solo sobre variables numéricas
X_num = df[num_cols]

# Detectar outliers
iso = IsolationForest(contamination=0.05, random_state=42)
outliers = iso.fit_predict(X_num)

# Eliminar outliers multivariados
df = df[outliers == 1]


In [11]:
# Identificar columnas categóricas
cat_cols = df.select_dtypes(include=['object']).columns

# Convertir a variables dummies
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Escalamiento de todas las variables numéricas excepto la clase
X_cols = df.drop(columns=['actual_productivity', 'actual_productivity_class']).columns
df[X_cols] = scaler.fit_transform(df[X_cols])


In [15]:
from sklearn.model_selection import train_test_split

# Features y target
X = df.drop(columns=['actual_productivity_class', 'actual_productivity'])
y = df['actual_productivity_class']

# Separar datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

In [23]:
# Modelos
models = {
    "k-NN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel='rbf', C=1),
    "LogReg": LogisticRegression(max_iter=1000),
    "DecisionTree": DecisionTreeClassifier(max_depth=5),
    "RandomForest": RandomForestClassifier(n_estimators=100),
    "NaiveBayes": GaussianNB()
}

# Entrenar y evaluar
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name}: Accuracy = {acc:.4f}")

k-NN: Accuracy = 0.9206
SVM: Accuracy = 0.8889
LogReg: Accuracy = 0.9365
DecisionTree: Accuracy = 0.9048
RandomForest: Accuracy = 0.8889
NaiveBayes: Accuracy = 0.9048


In [24]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Random Forest con Grid Search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5],
}
grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print("🎯 Mejor modelo (Grid Search - Random Forest):")
print("Accuracy:", accuracy_score(y_test, grid.predict(X_test)))
print("Mejores hiperparámetros:", grid.best_params_)

🎯 Mejor modelo (Grid Search - Random Forest):
Accuracy: 0.8888888888888888
Mejores hiperparámetros: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
